In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

from AnalysePoisson import *  
from AnalyseNegBinomial import *  

# Load data

In [ ]:
Filename='Gentamicin'
#Filename='Ciprofloxacin'
#Filename='Chloramphenicol'
#Filename='Tetracycline'

singleCellData = pd.read_csv(f'Tables/{Filename}_measured.csv',index_col=0)
# ensure all droplets that were labelled as thrash are removed
singleCellData.drop(singleCellData[singleCellData['trash']].index, inplace = True)


dataset=0
singleCellData=singleCellData[singleCellData.dataset==dataset]# code considers only a single dataset because we set the threshold per dataset of 6 chips
removal_threshold=20 # this is the same threshold to define sucessful growth and needs to be adated according to the table in the ESI

In [ ]:
singleCellData

In [ ]:
concentration = singleCellData.concentration.unique().tolist()

# Check if labeling missend any  outliers in number of initially detected cells: Makes computation more robust

<ol>
  <li> first automised overview of all concentrations and then look at the plots individually</li>
  <li> warning sometimes small counts cannot be seen by eye. So use extend of the x-axis to judge maximal count</li>
   <li> choose a removal threshold that works for all concentrations  </li>
  <li> c will be antibiotic concentration</li>
</ol> 

In [ ]:
Tot = np.size(concentration)
Cols = 2
# Compute Rows required
Rows = Tot // Cols 
Rows += Tot % Cols

# Create a Position index
Position = range(1,Tot + 1)

fig = plt.figure(figsize=(20, 10), dpi=80)
for k in range(Tot):
  # add every single subplot to the figure with a for loop

    foo=singleCellData[singleCellData.concentration==concentration[k]].n_cells.reset_index()
    df=foo.groupby(['n_cells']).count()
    df=df.rename(columns={"index": "count"})
    initial_cells=np.sort(foo.n_cells.unique().tolist())

    ax = fig.add_subplot(Rows,Cols,Position[k])
    ax.bar(initial_cells,df['count'])
    plt.title(f'[AB] = {concentration[k]}')
    plt.xlabel('initial detected cells')
    plt.ylabel('count')
    plt.rcParams.update({'font.size': 22})
    plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# look at a single panel:

c=0
foo=singleCellData[singleCellData.concentration==c].n_cells.reset_index()

df=foo.groupby(['n_cells']).count()
df=df.rename(columns={"index": "count"})

initial_cells=np.sort(foo.n_cells.unique().tolist())

plt.figure(figsize=(20, 10), dpi=80)
plt.bar(initial_cells,df['count'])
plt.rcParams.update({'font.size': 22})

plt.xlabel('initial detected cells')
plt.ylabel('count')
plt.tight_layout()

plt.show()

In [ ]:
singleCellDataAjusted=singleCellData[singleCellData['n_cells']<removal_threshold]

## Print how many measurments have been removed

In [ ]:
singleCellData.groupby(['concentration']).n_cells.count()-singleCellDataAjusted.groupby(['concentration']).n_cells.count()

# Classify droplets as positive or negative

<ol>
  <li> intentionally non automised so one looks at the plots individually</li>
  <li> c will be antibiotic concentration</li>
</ol> 

<ol>
  <li> classify according to threshold of intensity or initial  cell number  </li>
  <li> classify according to Gaussian mixture </li>
  <li> compare classifications </li>
</ol>

## Look at dependence on the initial number of cells

In [ ]:
ylim=[-5,80]

Tot = np.size(concentration)
Cols = 2
# Compute Rows required
Rows = Tot // Cols 
Rows += Tot % Cols

# Create a Position index
Position = range(1,Tot + 1)

fig = plt.figure(figsize=(20, 10), dpi=80)
for k in range(Tot):
  # add every single subplot to the figure with a for loop
    ax = fig.add_subplot(Rows,Cols,Position[k])
    
    sns.stripplot(ax=ax,data=singleCellDataAjusted[singleCellDataAjusted.concentration==concentration[k]], x="n_cells", y="n_cells_final",palette='bright')

    plt.title(f'[AB] = {concentration[k]}')
    plt.grid()

    plt.rcParams.update({'font.size': 22})
plt.tight_layout()
plt.show()

# Show threshold classification

In [ ]:
singleCellDataAjusted['dead']=singleCellDataAjusted.n_cells_final < removal_threshold
singleCellDataAjusted['dead'] =  singleCellDataAjusted.dead.replace({True: 1, False: 0})

# Analyse Chips

## Run Analysis

In [ ]:
chip_info = (
    singleCellDataAjusted
    .groupby('concentration')
    .apply(lambda x: AnalyseNegBinomChip(x))
    .droplevel(level=1)
)
'''
chip_info = (
    singleCellDataAjusted
    .groupby('AB')
    .apply(lambda x: AnalysePoissonianChip(x))
    .droplevel(level=1)
)
'''

In [ ]:
chip_info

In [ ]:
# add back meta data
metadata_cols = ['dataset', 'date', 'date_index']

metadata = (
    singleCellDataAjusted
    .groupby('concentration')[metadata_cols]
    .first()
)

chip_info = chip_info.join(metadata)
chip_info = chip_info.reset_index()

In [ ]:
chip_info

# Save Data

In [ ]:
'''
this creastes the table for the single dataset but the github provided them unified for convenience
'''
chip_info.to_csv(f'{Folder}chip_info_{Filename}_NegBinom_{dataset}.csv')